# BEED Dataset with TSFresh

This notebook demonstrates an ML workflow specifically using `tsfresh` to extract time-series features for the BEED dataset. Since the dataset does not have an explicit time or sequence ID column, we will artificially generate them to treat the rows as a single continuous sequential time series. We will use a rolling window to extract time-series features for out-of-sample forecasting, followed by hyperparameter tuning.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

import tsfresh
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import roll_time_series
from tsfresh.feature_extraction import MinimalFCParameters

## 1. Data Pre-Processing for TSFresh
`tsfresh` requires data in a specific long format with a distinct `id` for each time series and a `time` column. We will artificially construct an `id` and a sequential `time` index.

In [ ]:
# Load data
data_path = 'data/01_raw/beed/BEED_Data.csv'
df = pd.read_csv(data_path)

target_col = 'y'

# Set up tsfresh variables
df['id'] = 1
df['time'] = np.arange(len(df))

feature_cols = [col for col in df.columns if col not in [target_col, 'id', 'time']]
df_ts = df[['id', 'time'] + feature_cols]

display(df_ts.head())

,id,time,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16
0,1,0,4,7,18,25,28,27,20,10,-10,-18,-20,-16,13,32,12,10
1,1,1,87,114,120,106,76,54,28,5,-19,-49,-85,-102,-100,-89,-61,-21
2,1,2,-131,-133,-140,-131,-123,-108,-58,-51,-70,-77,-76,-76,-73,-57,-40,-14
3,1,3,68,104,73,34,-12,-26,-38,-36,-67,-88,-25,31,18,-4,6,-29
4,1,4,-67,-90,-97,-94,-86,-71,-43,-11,23,46,58,50,39,19,-9,-41


## 2. Rolling Time Series & Feature Extraction
To predict failure at time `T` using the sequence of past events, we create rolling windows. We will use a `max_timeshift` of 4 (so each window looks at the current step and up to 4 past steps) to avoid exploding the dataset size.

In [9]:
print("Rolling time series...")
# Create rolling windows
df_rolled = roll_time_series(df_ts, column_id='id', column_sort='time', 
                             max_timeshift=4, min_timeshift=0)

print("Extracting features...")
# We use MinimalFCParameters to keep extraction time extremely fast (extracts mean, median, min, max, std, variance, etc.)
# For deeper features, you could use EfficientFCParameters()
extraction_settings = MinimalFCParameters()
X_tsfresh = extract_features(df_rolled, column_id='id', column_sort='time', 
                             default_fc_parameters=extraction_settings, disable_progressbar=False)

# The index of X_tsfresh is a tuple of (id, time) generated by roll_time_series.
# We extract just the 'time' part of the tuple to align with our target variable.
X_tsfresh.index = [idx[1] for idx in X_tsfresh.index]
X_tsfresh.index.name = 'time'

display(X_tsfresh.head())

Rolling time series...


Rolling: 100%|██████████| 40/40 [00:14<00:00,  2.84it/s]


Extracting features...


Feature Extraction: 100%|██████████| 40/40 [00:34<00:00,  1.15it/s]


,X1__sum_values,X1__median,X1__mean,X1__length,X1__standard_deviation,X1__variance,X1__root_mean_square,X1__maximum,X1__absolute_maximum,X1__minimum,...,X16__sum_values,X16__median,X16__mean,X16__length,X16__standard_deviation,X16__variance,X16__root_mean_square,X16__maximum,X16__absolute_maximum,X16__minimum
time,,,,,,,,,,,,,,,,,,,,,
0,4.0,4.0,4.000000,1.0,0.000000,0.000000,4.000000,4.0,4.0,4.0,...,10.0,10.0,10.000000,1.0,0.000000,0.000000,10.000000,10.0,10.0,10.0
1,91.0,45.5,45.500000,2.0,41.500000,1722.250000,61.583277,87.0,87.0,4.0,...,-11.0,-5.5,-5.500000,2.0,15.500000,240.250000,16.446884,10.0,21.0,-21.0
2,-40.0,4.0,-13.333333,3.0,89.838126,8070.888889,90.822171,87.0,131.0,-131.0,...,-25.0,-14.0,-8.333333,3.0,13.274872,176.222222,15.673757,10.0,21.0,-21.0
3,28.0,36.0,7.000000,4.0,85.401991,7293.500000,85.688389,87.0,131.0,-131.0,...,-54.0,-17.5,-13.500000,4.0,14.568802,212.250000,19.862024,10.0,29.0,-29.0
4,-39.0,4.0,-7.800000,5.0,81.920449,6710.960000,82.290947,87.0,131.0,-131.0,...,-95.0,-21.0,-19.000000,5.0,17.052859,290.800000,25.530374,10.0,41.0,-41.0


## 3. Train/Test Split
**CRITICAL FIX**: Unlike the predictive maintenance dataset, the BEED dataset is ordered sequentially by its classes (e.g. all class 1s first, then all class 2s). Doing a temporal un-shuffled split caused the test set to consist entirely of a single class that the model never saw during training! We must shuffle and stratify the split to ensure all classes are represented in both train and test.

In [10]:
# Align target with extracted features
y = df.set_index('time')[target_col].loc[X_tsfresh.index]

# Impute any potential missing values (tsfresh can sometimes output NaNs for variances of length 1, etc.)
X_tsfresh = X_tsfresh.fillna(0)

# Random split (80/20) with stratification to handle the ordered dataset
X_train, X_test, y_train, y_test = train_test_split(X_tsfresh, y, test_size=0.2, shuffle=True, stratify=y, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (6400, 160)
Testing set shape: (1600, 160)


## 4. Hyperparameter Tuning & Model Training
Since we are now using a shuffled dataset, we will use a standard `StratifiedKFold` instead of `TimeSeriesSplit` for cross-validation.

In [11]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42)

print("Starting Hyperparameter Tuning...")
# Using f1_macro to safely handle multi-class targets
grid_search = GridSearchCV(rf, param_grid, cv=skf, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")

Starting Hyperparameter Tuning...
Fitting 3 folds for each of 18 candidates, totalling 54 fits

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}


## 5. Evaluation

In [12]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       0.99      0.98      0.99       400
           2       0.97      0.99      0.98       400
           3       0.99      0.96      0.98       400

    accuracy                           0.99      1600
   macro avg       0.99      0.99      0.99      1600
weighted avg       0.99      0.99      0.99      1600

